<a href="https://colab.research.google.com/github/Decoding-Data-Science/airesidency/blob/main/MC11_Beginner_HR_LoRA_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MC11 | From HR examples to a small fine-tune
**Decoding Data Science · Beginner live workshop · Google Colab**

**Big idea:** a fine-tune learns *how to answer* from examples; a retrieval system supplies changing policies. Today we clean fictional HR questions, watch the tokenizer, then train a small LoRA adapter on Qwen2.5-0.5B-Instruct. This is an educational demonstration, not a production HR assistant. The synthetic portal steps and policies are invented - do not rely on them as UAE law or your employer policy.

**Run order:** Runtime > Change runtime type > T4 GPU if offered. Then Runtime > Run all, or execute cells in order. Free GPU access and run time are not guaranteed. Without a GPU, complete the data prep, then skip the training section. All datasets and models are downloaded over the internet. No paid API key is needed. A public model normally loads without `HF_TOKEN`; if you use a token, add it as a *Colab Secret*, never paste it into the notebook.


## Step 0 - Add your Hugging Face token to Colab Secrets

1. Get a token at https://huggingface.co/settings/tokens (a **Read** token is enough; use **Write** only if you want the optional push to the Hub at the end).
2. In Colab, click the **key icon (Secrets)** in the left sidebar.
3. Click **Add new secret**. Name: `HF_TOKEN`. Value: your token.
4. Turn on **Notebook access** for this notebook.

Your token now lives in Colab, not in this notebook. You can share the notebook without leaking it.

In [3]:
%pip -q install -U "transformers>=4.45,<5" "datasets>=3,<5"


In [3]:
# Optional Hugging Face token from Colab Secrets. Never print or save its value.
import os
HF_TOKEN = None
try:
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        print("Optional HF_TOKEN unavailable. Public-model steps can continue.")
except ImportError:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("Optional Hugging Face token is available (value hidden).")
else:
    print("Continuing with public resources, no token required.")


Optional Hugging Face token is available (value hidden).


In [4]:
# Optional: confirm the token works
if HF_TOKEN:
    from huggingface_hub import whoami
    print("Logged in to Hugging Face as:", whoami(token=HF_TOKEN)["name"])

Logged in to Hugging Face as: decodingdatascience


## Step 1 - Watch it happen: text -> IDs -> text
One tokenizer per model. It ships with the weights.

In [5]:
from transformers import AutoTokenizer

tok_gpt2 = AutoTokenizer.from_pretrained("gpt2")

text = "كم يوم إجازة سنوية أحصل عليها؟"

ids    = tok_gpt2.encode(text)                # text -> token IDs
pieces = tok_gpt2.convert_ids_to_tokens(ids)  # the chunks the model sees
back   = tok_gpt2.decode(ids)                 # IDs -> exact original text

print(len(ids), "tokens")
print(ids)
print(pieces)
print("Round trip OK:", back == text)

35 tokens
[149, 225, 25405, 18923, 232, 30335, 25405, 17550, 98, 148, 105, 34247, 110, 45632, 17550, 111, 23338, 30335, 22654, 45632, 17550, 96, 148, 255, 148, 113, 13862, 17550, 117, 13862, 22654, 29519, 12919, 148, 253]
['Ù', 'ĥ', 'Ùħ', 'ĠÙ', 'Ĭ', 'ÙĪ', 'Ùħ', 'ĠØ', '¥', 'Ø', '¬', 'Ø§Ø', '²', 'Ø©', 'ĠØ', '³', 'ÙĨ', 'ÙĪ', 'ÙĬ', 'Ø©', 'ĠØ', '£', 'Ø', 'Ń', 'Ø', 'µ', 'ÙĦ', 'ĠØ', '¹', 'ÙĦ', 'ÙĬ', 'Ùĩ', 'Ø§', 'Ø', 'Ł']
Round trip OK: True


**Try it:** change `text` to Arabic, a phone number, some Python code, or a typo. Watch the token count jump.
The `Ġ` symbol is GPT-2's marker for a leading space.

## Step 2 - Same text, different tokenizers
Vocabularies freeze at training time. Use the tokenizer that ships with your model - never mix and match.

In [6]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # small, open, has a chat template - our model for the rest of the notebook
tok = AutoTokenizer.from_pretrained(MODEL_ID)

samples = [
    "Tokenization decides what your model can learn.",
    "How many days of annual leave do I get in the UAE?",
    "كم يوم إجازة سنوية أحصل عليها؟",
    "def add(a, b): return a + b",
]
tokenizers = {"gpt2": tok_gpt2, MODEL_ID: tok}

for s in samples:
    print(repr(s))
    for name, t in tokenizers.items():
        print(f"   {name:<30} {len(t.encode(s)):>3} tokens")
    print()

print("Vocab sizes:", {name: len(t) for name, t in tokenizers.items()})

'Tokenization decides what your model can learn.'
   gpt2                             9 tokens
   Qwen/Qwen2.5-0.5B-Instruct       9 tokens

'How many days of annual leave do I get in the UAE?'
   gpt2                            13 tokens
   Qwen/Qwen2.5-0.5B-Instruct      13 tokens

'كم يوم إجازة سنوية أحصل عليها؟'
   gpt2                            35 tokens
   Qwen/Qwen2.5-0.5B-Instruct      11 tokens

'def add(a, b): return a + b'
   gpt2                            11 tokens
   Qwen/Qwen2.5-0.5B-Instruct      10 tokens

Vocab sizes: {'gpt2': 50257, 'Qwen/Qwen2.5-0.5B-Instruct': 151665}


In [7]:
for s in samples:
    print(f"\nText: {repr(s)}")
    ids_qwen = tok.encode(s)
    pieces_qwen = tok.convert_ids_to_tokens(ids_qwen)
    print("  Token split (Qwen):", pieces_qwen)
    print("  Token IDs (Qwen):", ids_qwen)


Text: 'Tokenization decides what your model can learn.'
  Token split (Qwen): ['Token', 'ization', 'Ġdecides', 'Ġwhat', 'Ġyour', 'Ġmodel', 'Ġcan', 'Ġlearn', '.']
  Token IDs (Qwen): [3323, 2022, 27627, 1128, 697, 1614, 646, 3960, 13]

Text: 'How many days of annual leave do I get in the UAE?'
  Token split (Qwen): ['How', 'Ġmany', 'Ġdays', 'Ġof', 'Ġannual', 'Ġleave', 'Ġdo', 'ĠI', 'Ġget', 'Ġin', 'Ġthe', 'ĠUAE', '?']
  Token IDs (Qwen): [4340, 1657, 2849, 315, 9775, 5274, 653, 358, 633, 304, 279, 46849, 30]

Text: 'كم يوم إجازة سنوية أحصل عليها؟'
  Token split (Qwen): ['ÙĥÙħ', 'ĠÙĬÙĪÙħ', 'ĠØ¥', 'Ø¬Ø§Ø²', 'Ø©', 'ĠØ³ÙĨ', 'ÙĪÙĬØ©', 'ĠØ£', 'ØŃØµÙĦ', 'ĠØ¹ÙĦÙĬÙĩØ§', 'ØŁ']
  Token IDs (Qwen): [124613, 128587, 85153, 130707, 25871, 126760, 126395, 63415, 130317, 129387, 128332]

Text: 'def add(a, b): return a + b'
  Token split (Qwen): ['def', 'Ġadd', '(a', ',', 'Ġb', '):', 'Ġreturn', 'Ġa', 'Ġ+', 'Ġb']
  Token IDs (Qwen): [750, 912, 2877, 11, 293, 1648, 470, 264, 488, 293]


## Step 3 - The chat template: the silent killer
Instruct models are trained on a very specific text format. If your fine-tuning data does not use the **model's own** template,
the model learns the wrong format - it rambles, never stops, or ignores the system prompt.

In [8]:
messages = [
    {"role": "system",    "content": "You are a helpful HR assistant."},
    {"role": "user",      "content": "How do I apply for leave?"},
    {"role": "assistant", "content": "Submit a request in the HR portal at least two weeks ahead, and copy your manager."},
]

rendered = tok.apply_chat_template(messages, tokenize=False)
print(rendered)

<|im_start|>system
You are a helpful HR assistant.<|im_end|>
<|im_start|>user
How do I apply for leave?<|im_end|>
<|im_start|>assistant
Submit a request in the HR portal at least two weeks ahead, and copy your manager.<|im_end|>



In [9]:
# The prompt part only, ending exactly where the model should start answering
prompt_only = tok.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
print(prompt_only)
print("The full text starts with the prompt:", rendered.startswith(prompt_only))

<|im_start|>system
You are a helpful HR assistant.<|im_end|>
<|im_start|>user
How do I apply for leave?<|im_end|>
<|im_start|>assistant

The full text starts with the prompt: True


## Step 4 - Build a small HR-assistant dataset (JSONL)
One JSON object per line, each with a `messages` list. We deliberately include some **bad rows** so the cleaning steps have something to catch.

In [10]:
import json

SYSTEM = "You are a helpful HR assistant for a UAE company."

def row(q, a):
    return {"messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": q},
        {"role": "assistant", "content": a},
    ]}

raw_rows = [
    row("How do I apply for annual leave?", "Submit a leave request in the HR portal at least two weeks before your start date and copy your line manager."),
    row("Where can I download my salary certificate?", "Go to HR portal > Documents > Salary Certificate, choose the purpose, and download the PDF. It is ready within one working day."),
    row("Who do I contact about a payroll error?", "Email the payroll team with your employee ID, the month affected, and a short description of the error."),
    row("Can I work from home on Fridays?", "Remote Fridays depend on your team's policy. Check with your line manager, then log it in the HR portal."),
    row("How do I update my emergency contact?", "Open your profile in the HR portal, choose Emergency Contacts, and save the new details."),
    row("What should I do on my first day?", "Collect your access card from reception, complete the IT setup checklist, and join the 10 AM onboarding session."),
    # exact duplicate
    row("How do I apply for annual leave?", "Submit a leave request in the HR portal at least two weeks before your start date and copy your line manager."),
    # near duplicate (extra spaces, different case)
    row("how do I  apply for annual leave?", "Submit a leave request in the HR portal at least two weeks before your start date and copy your line manager."),
    # contains PII - must be filtered
    row("What is Sara's phone number?", "You can reach Sara on +971 50 123 4567 or sara.k@example.com."),
    # empty answer - must be filtered
    row("How do I claim medical expenses?", ""),
    # boilerplate that needs cleaning
    row("Is there a dress code?", "Business casual Monday to Thursday, smart casual on Friday.\n\nSent from my iPhone"),
]

with open("hr_faq.jsonl", "w", encoding="utf-8") as f:
    for r in raw_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
    f.write('{"messages": [ this line is broken json\n')   # a corrupt line, on purpose

print(open("hr_faq.jsonl", encoding="utf-8").read()[:600], "...")

{"messages": [{"role": "system", "content": "You are a helpful HR assistant for a UAE company."}, {"role": "user", "content": "How do I apply for annual leave?"}, {"role": "assistant", "content": "Submit a leave request in the HR portal at least two weeks before your start date and copy your line manager."}]}
{"messages": [{"role": "system", "content": "You are a helpful HR assistant for a UAE company."}, {"role": "user", "content": "Where can I download my salary certificate?"}, {"role": "assistant", "content": "Go to HR portal > Documents > Salary Certificate, choose the purpose, and downloa ...


## Step 5 - Validate every line
Loss spikes and silent failures often come from a handful of corrupt rows. Check the schema before anything else.

In [11]:
VALID_ROLES = {"system", "user", "assistant"}

def validate(obj):
    msgs = obj.get("messages")
    if not isinstance(msgs, list) or len(msgs) < 2:
        return "missing or too-short 'messages'"
    for m in msgs:
        if not isinstance(m, dict) or m.get("role") not in VALID_ROLES or not isinstance(m.get("content"), str):
            return "bad message shape or role"
    if msgs[-1]["role"] != "assistant":
        return "last message is not from the assistant"
    return None

valid, rejected = [], []
with open("hr_faq.jsonl", encoding="utf-8") as f:
    for n, line in enumerate(f, 1):
        try:
            obj = json.loads(line)
        except json.JSONDecodeError as e:
            rejected.append((n, f"invalid JSON: {e.msg}"))
            continue
        err = validate(obj)
        (rejected.append((n, err)) if err else valid.append(obj))

print(f"valid: {len(valid)}   rejected: {len(rejected)}")
for n, why in rejected:
    print(f"  line {n}: {why}")

valid: 11   rejected: 1
  line 12: invalid JSON: Expecting value


## Step 6 - Clean, deduplicate, filter
Pipeline: raw -> **clean -> deduplicate -> filter** -> format -> tokenize & split. Each step is a function you can re-run.

In [12]:
import re

BOILERPLATE = [r"Sent from my iPhone", r"Best regards,.*$", r"This email is confidential.*$"]

def clean_text(s):
    for pat in BOILERPLATE:
        s = re.sub(pat, "", s, flags=re.IGNORECASE | re.DOTALL)
    s = s.replace("\u00a0", " ")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def clean_row(obj):
    return {"messages": [{"role": m["role"], "content": clean_text(m["content"])} for m in obj["messages"]]}

def dedup_key(obj):
    # normalize case and whitespace so near duplicates collide
    return " || ".join(re.sub(r"\s+", " ", m["content"].lower()).strip() for m in obj["messages"])

PII_PATTERNS = {
    "email": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "phone": r"\+?\d[\d\s-]{7,}\d",
    "api_key": r"(hf_[A-Za-z0-9]{20,}|sk-[A-Za-z0-9]{20,})",
}

def filter_reason(obj):
    answer = obj["messages"][-1]["content"]
    if len(answer) < 10:
        return "answer too short"
    full = " ".join(m["content"] for m in obj["messages"])
    for name, pat in PII_PATTERNS.items():
        if re.search(pat, full):
            return f"contains {name}"
    return None

cleaned = [clean_row(o) for o in valid]

seen, deduped = set(), []
for o in cleaned:
    k = dedup_key(o)
    if k not in seen:
        seen.add(k)
        deduped.append(o)

final_rows, dropped = [], []
for o in deduped:
    why = filter_reason(o)
    (dropped.append((o["messages"][1]["content"], why)) if why else final_rows.append(o))

print(f"valid {len(valid)} -> cleaned {len(cleaned)} -> deduped {len(deduped)} -> kept {len(final_rows)}")
for q, why in dropped:
    print(f"  dropped: {q!r} ({why})")

with open("hr_faq_clean.jsonl", "w", encoding="utf-8") as f:
    for o in final_rows:
        f.write(json.dumps(o, ensure_ascii=False) + "\n")

valid 11 -> cleaned 11 -> deduped 9 -> kept 7
  dropped: "What is Sara's phone number?" (contains email)
  dropped: 'How do I claim medical expenses?' (answer too short)


## Step 7 - Tokenize the dataset and mask the prompt
**The -100 trick:** label `-100` is ignored by the loss. Prompt tokens get `-100`; answer tokens keep their IDs.
The model learns to **answer**, not to echo the question.

In [13]:
from datasets import load_dataset

MAX_LEN = 1024
ds = load_dataset("json", data_files="hr_faq_clean.jsonl", split="train")

def mask_prompt(input_ids, prompt_len):
    # keep labels only for the assistant's answer
    return [-100] * prompt_len + input_ids[prompt_len:]

def to_features(row):
    text   = tok.apply_chat_template(row["messages"], tokenize=False)
    prompt = tok.apply_chat_template(row["messages"][:-1], tokenize=False, add_generation_prompt=True)
    enc    = tok(text, truncation=True, max_length=MAX_LEN, add_special_tokens=False)
    prompt_len = min(len(tok(prompt, add_special_tokens=False)["input_ids"]), len(enc["input_ids"]))
    enc["labels"] = mask_prompt(enc["input_ids"], prompt_len)
    return enc

tokenized = ds.map(to_features, remove_columns=ds.column_names)
print(tokenized)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 7
})


In [14]:
# See the mask with your own eyes: what does the loss actually train on?
ex = tokenized[0]
trained_ids = [i for i, l in zip(ex["input_ids"], ex["labels"]) if l != -100]

print("Tokens in example:", len(ex["input_ids"]), "| masked (prompt):", ex["labels"].count(-100), "| trained on:", len(trained_ids))
print("\n--- The model is trained to produce ONLY this: ---")
print(tok.decode(trained_ids))

Tokens in example: 56 | masked (prompt): 32 | trained on: 24

--- The model is trained to produce ONLY this: ---
Submit a leave request in the HR portal at least two weeks before your start date and copy your line manager.<|im_end|>



## Step 8 - Length check and train / validation split
Truncated examples teach the model to stop mid-sentence. Look at the lengths before you train.

In [15]:
lengths = [len(x) for x in tokenized["input_ids"]]
print(f"examples: {len(lengths)}  min: {min(lengths)}  max: {max(lengths)}  mean: {sum(lengths)/len(lengths):.0f}")
print("hit MAX_LEN (truncated):", sum(l >= MAX_LEN for l in lengths))

splits = tokenized.train_test_split(test_size=0.2, seed=42)
print(splits)

splits.save_to_disk("hr_faq_tokenized")
print("Saved to ./hr_faq_tokenized")

examples: 7  min: 43  max: 61  mean: 55
hit MAX_LEN (truncated): 0
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 5
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2
    })
})


Saving the dataset (0/1 shards):   0%|          | 0/5 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2 [00:00<?, ? examples/s]

Saved to ./hr_faq_tokenized


## Where do training examples come from?
**Synthetic HR starter data** is generated in Step 4. It is intentionally tiny. Use it to test the mechanics, never to claim domain accuracy. For real enterprise work, use approved, anonymized HR tickets and verified policy answers, with permission from the data owner. Keep changing policy text in a retrieval source rather than trying to memorize it in an adapter.

**Public practice data** can teach the format, but not your company's policy:
- [Databricks Dolly 15k](https://huggingface.co/datasets/databricks/databricks-dolly-15k) has `instruction`, `context`, `response`, `category` fields. The optional cell below samples up to 200 rows and maps them to chat messages. Its license and suitability for your use must be checked on its dataset card.
- [No Robots](https://huggingface.co/datasets/HuggingFaceH4/no_robots) offers human-written instruction/chat data. Inspect its card and fields before mixing it with HR examples.

Never put staff records, customer chats, employee IDs, phone numbers, keys, or private policies on a public hub. The regex below is a *classroom illustration*, not a sufficient privacy control.

In [16]:
# OPTIONAL: sample public instruction data (NOT company HR policy). This does not affect the HR training below.
LOAD_DOLLY = False  # change to True to download a small, reproducible sample
if LOAD_DOLLY:
    from datasets import load_dataset
    dolly = load_dataset("databricks/databricks-dolly-15k", split="train", token=HF_TOKEN)
    dolly_small = dolly.shuffle(seed=42).select(range(min(200, len(dolly))))
    def dolly_to_chat(x):
        question = x["instruction"]
        if x.get("context", "").strip():
            question += "\n\nContext: " + x["context"]
        return {"messages": [
            {"role": "user", "content": question},
            {"role": "assistant", "content": x["response"]},
        ]}
    dolly_chat = dolly_small.map(dolly_to_chat, remove_columns=dolly_small.column_names)
    print("Sample rows:", len(dolly_chat))
    print("Example (not a DDS HR policy):", dolly_chat[0]["messages"])
else:
    print("Dolly download skipped. Set LOAD_DOLLY=True to explore 200 examples.")


Dolly download skipped. Set LOAD_DOLLY=True to explore 200 examples.


## Pre-flight checklist - before you press Train

- [ ] Schema validates on every single line
- [ ] Chat template matches the base model exactly
- [ ] No exact or near duplicates
- [ ] No PII, API keys or secrets anywhere (your token lives in Colab Secrets, not in the data or the notebook)
- [ ] Length distribution checked - nothing important truncated
- [ ] Prompt tokens masked with -100
- [ ] Validation split held out and never trained on

**Homework:** replace the sample rows with 30 real questions from your own domain and run the whole notebook again.

---
Decoding Data Science · MC11 · AI Residency

## Step 9 | A tiny LoRA training run on free Colab GPU
**Option choice:** Axolotl on JarvisLabs is useful for bigger experiments but adds setup and rental cost. Here we use `transformers` + PEFT LoRA with one small Qwen instruct model. LoRA trains small adapter matrices while freezing the original model. No OpenAI API or embedding is needed: embeddings support *retrieval*, not this supervised training step. If Colab has no GPU today, teach through Step 8 and run this later. **This demo is tiny:** 6-8 synthetic rows can show that loss falls, but cannot prove a reliable HR assistant. Before production, use a larger reviewed dataset and a separate holdout evaluation.

In [1]:
# Install the training library after the data-prep installs above.
%pip -q install -U "transformers>=4.45,<5" "peft>=0.13,<1" "accelerate>=1,<2"
%pip -q install --upgrade torchao
import torch, transformers, peft
print("torch",torch.__version__,"transformers",transformers.__version__,"peft",peft.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("Training needs a Colab GPU. Runtime > Change runtime type > T4 GPU (if available); rerun from the top. Data-prep cells work without it.")
print("GPU:", torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 103.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
torch 2.11.0+cu128 transformers 4.57.6 peft 0.21.0
GPU: Tesla T4


In [21]:
import torchao
print(f"torchao version: {torchao.__version__}")

torchao version: 0.10.0


In [23]:
!pip install -U torchao==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [1]:
import torchao
print(f"torchao version: {torchao.__version__}")

torchao version: 0.16.0


### Check a held-out question before training
We do **not** train on this question. Record the exact generated answer to compare later. There is no expected success from so few invented examples. A new question is not enough to measure HR accuracy; evaluate against verified policy answers.

In [17]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
import torch
# Explicit GPU requirement, small model, half precision. Colab RAM and GPU quota vary.
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, low_cpu_mem_usage=True, token=HF_TOKEN)
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()  # frozen embeddings still need a gradient path with checkpointing
model = get_peft_model(model, LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"], bias="none", task_type="CAUSAL_LM"
))
model.print_trainable_parameters()
model.to("cuda")

def ask(model, question):
    model.eval()
    prompt = tok.apply_chat_template([
        {"role":"system", "content": SYSTEM},
        {"role":"user", "content": question},
    ], tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=60, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

TEST_QUESTION = "What should I do if I cannot access the HR portal?"
print("BEFORE (actual generated text):", ask(model, TEST_QUESTION))

`torch_dtype` is deprecated! Use `dtype` instead!


trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


BEFORE (actual generated text): If you cannot access the HR portal, there are several steps you can take to troubleshoot and resolve the issue:

1. Check your internet connection: Ensure that your device is connected to the internet and that it has proper connectivity.

2. Restart your device: Sometimes, simply restarting your device can resolve


### Train on the assistant answer, not the prompt
The earlier `labels` use -100 on the prompt. The custom collator pads input IDs and attention masks, with -100 for padding labels too. Keep the model's own chat template. We use short examples and a few update steps to fit a beginner workshop. Lower loss only means the model fitted this small dataset; it is *not* an HR quality score.

In [18]:
from transformers import Trainer, TrainingArguments
# Use the cleaned, masked Dataset created in Step 7. No manual data upload.
assert len(tokenized) >= 4, "Run Step 7 first and check the cleaned examples."
assert all(any(v != -100 for v in row["labels"]) for row in tokenized), "No assistant tokens to train."
tok.padding_side = "right"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model.config.pad_token_id = tok.pad_token_id

def collate(features):
    inputs = [{"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]} for f in features]
    batch = tok.pad(inputs, padding=True, return_tensors="pt")
    width = batch["input_ids"].shape[1]
    batch["labels"] = torch.tensor([f["labels"] + [-100] * (width - len(f["labels"])) for f in features])
    return batch

args = TrainingArguments(
    output_dir="./mc11-hr-lora-demo", per_device_train_batch_size=1,
    gradient_accumulation_steps=2, max_steps=12, learning_rate=2e-4,
    logging_steps=2, save_strategy="no", report_to="none",
    fp16=True, gradient_checkpointing=True, remove_unused_columns=False,
)
trainer = Trainer(model=model, args=args, train_dataset=tokenized, data_collator=collate)
result = trainer.train()
print("Actual final training loss:", result.training_loss)


You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
2,3.413100
4,2.764900
6,2.859100
8,3.188400
10,2.676000
12,3.034000


Actual final training loss: 2.98927104473114


In [19]:
# Compare real model output, then save only the small adapter to Colab's temporary storage.
model.config.use_cache = True
print("AFTER (actual generated text):", ask(model, TEST_QUESTION))
model.save_pretrained("./mc11-hr-lora-demo/adapter")
tok.save_pretrained("./mc11-hr-lora-demo/adapter")
print("Adapter saved locally. Colab files vanish when the runtime resets; download it yourself if needed.")


AFTER (actual generated text): If you cannot access the HR portal, there are several steps you can take to troubleshoot and resolve the issue:

1. Check your internet connection: Ensure that your internet connection is stable and working properly.

2. Restart your computer: Sometimes, restarting your computer can help resolve connectivity issues.

3.
Adapter saved locally. Colab files vanish when the runtime resets; download it yourself if needed.


## Debrief: what did we actually learn?
- We observed the model's token IDs, formatted chat messages, cleaned fictional HR examples, masked the prompt, and ran LoRA on a 0.5B instruct model.
- A falling training loss is a plumbing check, not proof that the model knows policy. Tiny examples can overfit; no employee should receive unsupervised HR advice from this demo.
- For current HR policy, use permissioned document retrieval and cite the retrieved version. For stable response style, fine-tune on reviewed examples. Measure both separately with real holdout questions and human review.
- If GPU is unavailable: teach the data pipeline and use the optional Dolly cell. Do not claim the training ran.

**References:** [Qwen model card](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) · [PEFT quicktour](https://huggingface.co/docs/peft/quicktour) · [Transformers Trainer](https://huggingface.co/docs/transformers/main_classes/trainer) · [Dolly dataset](https://huggingface.co/datasets/databricks/databricks-dolly-15k).